## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. Just press ▶ on the cell below and wait
for the green **✅ Setup complete**, then run the rest top to bottom.

When it asks to **connect Google Drive**, click **Connect** — that lets the data
file download **only once** (it's saved to your Drive and reused by every
notebook) and saves your figures for your poster. You *can* skip it, but then each
notebook re-downloads the ~470 MB data and your figures won't be saved.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os

print("1/3  installing libraries ...")
get_ipython().system('pip install -q "mne==1.10.1" gdown')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

# Connect Drive so the data is downloaded ONCE (saved to your Drive) and your
# figures persist. If you skip it, we fall back to temporary storage.
print("3/3  connecting Google Drive ...")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = "/content/drive/MyDrive/DecodingBrain_data"
    os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
    saved = True
except Exception:
    data_dir = "data"                     # temporary (re-downloads each session)
    os.environ["CAMP_OUTPUT_DIR"] = "outputs"
    saved = False

os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, "synapse_preprocessed.pkl")
os.environ["CAMP_DATA_PATH"] = data_path

if os.path.exists(data_path):
    print("     data already saved in your Drive — skipping download \u26a1")
else:
    print("     downloading the data (~470 MB, one time only) ...")
    import gdown
    gdown.download(id="1Z-NENlKMjL-kL-N46lQ8QA1AbGM7bJHY", output=data_path, quiet=False)

print("\n\u2705 Setup complete.",
      "Data + figures are saved in your Drive (DecodingBrain_*)." if saved
      else "Heads up: you skipped Drive, so the data re-downloads each session.")


# Week 3 · Tier 3 — Is It Real? Leakage, Permutation & ROC

In Notebook 13 you engineered a classifier up to an **AUC ~0.81**. Exciting — but
a good scientist immediately asks two skeptical questions:

1. **Could that score happen by luck** with only 28 people? → **permutation test**
2. **Did we cheat** by choosing those features after peeking at the answers? →
   the **leakage trap** and its fix, **nested feature selection**

Then we'll draw the **ROC curve** that goes on every ML poster.

### By the end of this notebook you will be able to
1. Build a "null distribution" by shuffling labels and get a permutation p-value
2. *Demonstrate* data leakage producing a fake-great score
3. Fix it with in-fold (nested) feature selection
4. Draw and read an ROC curve

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score, roc_curve
import camp_utils as cu

data = cu.load_camp_data(verbose=False)

# rebuild the best model from Notebook 13 (the study's 5 features)
X, y, names = cu.build_psd_feature_matrix(data, cu.PAPER_PSD5)
real_auc = cu.loocv_auc(X, y)
print("Best model features:", names)
print(f"Real model AUC = {real_auc:.3f}")

## 1. The permutation test
Imagine we **shuffle** the EXP/CTRL labels — now they're random nonsense. Whatever
AUC the model gets on shuffled labels is **pure luck**. Do this hundreds of times
to build a distribution of "luck" AUCs (the **null distribution**), then ask: *is
our real AUC clearly bigger than the lucky ones?*

### ✏️ Your turn #1 — shuffle once
Make a shuffled copy of `y` and score the model on it (should be near 0.5).

In [ ]:
rng = np.random.default_rng(0)
# TODO: y_shuffled = a randomly permuted copy of y   (hint: rng.permutation(y))
y_shuffled = None

if y_shuffled is not None:
    print(f"AUC with shuffled labels = {cu.loocv_auc(X, y_shuffled):.3f}  (≈ chance)")
cu.check(y_shuffled is not None and sorted(y_shuffled) == sorted(y),
         "You shuffled the labels (same counts, random order).",
         "Use rng.permutation(y).")

## 2. Build the null distribution
Repeat the shuffle many times. We use 200 here (the real study uses 1000). This
takes a minute — that's normal.

In [ ]:
n_permutations = 200
null_aucs = []
for i in range(n_permutations):
    null_aucs.append(cu.loocv_auc(X, rng.permutation(y)))
    if (i + 1) % 50 == 0:
        print(f"  ...{i+1}/{n_permutations}")
null_aucs = np.array(null_aucs)
print(f"\nNull AUCs: mean = {null_aucs.mean():.3f}, max = {null_aucs.max():.3f}")

## 3. The permutation p-value
The p-value is the fraction of shuffled (luck) AUCs that matched or beat the real
one. The "+1"s keep it from being exactly 0.

In [ ]:
n_better = int(np.sum(null_aucs >= real_auc))
p_perm = (n_better + 1) / (n_permutations + 1)
print(f"Real AUC = {real_auc:.3f};  {n_better}/{n_permutations} shuffles did as well.")
print(f"Permutation p-value = {p_perm:.3f}")
print("Beats chance at 0.05?", "YES ✅" if p_perm < 0.05 else "not clearly")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(null_aucs, bins=20, color="lightgray", edgecolor="gray", label="shuffled (luck)")
ax.axvline(real_auc, color=cu.EXP_COLOR, linewidth=3, label=f"real model ({real_auc:.2f})")
ax.axvline(0.5, color="black", linestyle="--", linewidth=0.8, label="chance")
ax.set_xlabel("AUC"); ax.set_ylabel("Count")
ax.set_title(f"Our model vs. luck   (p = {p_perm:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig(cu.save_path("tier3_permutation.png"), dpi=300, bbox_inches="tight")
plt.show()

If the orange line sits to the **right** of the gray pile, the model found a real
signal — not luck. Good. Now the second skeptical question…

## 4. The leakage trap 🪤 (this one is sneaky)
In Notebook 13 we *knew* which 5 features to use. But what if you **don't** know,
and you let the computer **pick the best features from a big pile**? The catch:
*when* you pick matters enormously.

Let's build a **big, mostly-noisy pile** of 100 candidate features (every task ×
every time window × every band) and select the "best" ones two different ways.

In [ ]:
big_specs = [(t, p, b)
             for t in ["pmt", "let", "hlt", "ast"]
             for p in ["early_stim", "mid_stim", "late_stim", "full_stim", "poststim"]
             for b in cu.BAND_ORDER]
Xbig, ybig, _ = cu.build_psd_feature_matrix(data, big_specs)
print(f"Candidate pile: {Xbig.shape[1]} features (most are noise), n = {len(ybig)}")

**Way A — the WRONG way (leakage):** look at *all* the labels, keep the 5 features
that best separate the groups, *then* run LOOCV on those 5.

In [ ]:
def loocv_leaky(X, y, k=5):
    selector = SelectKBest(f_classif, k=k).fit(X, y)   # ← peeks at ALL labels!
    return cu.loocv_auc(selector.transform(X), y)

auc_leaky = loocv_leaky(Xbig, ybig)
print(f"Way A (leaky — select using all data):  AUC = {auc_leaky:.3f}  😍 looks amazing!")

**Way B — the RIGHT way (nested):** inside each fold, select the best features
using **only the training subjects**, then predict the held-out one. The test
subject never influences which features we keep.

### ✏️ Your turn #2 — fix the leak
Complete the in-fold selection.

In [ ]:
def loocv_nested(X, y, k=5):
    n = len(y)
    proba = np.full(n, np.nan)
    for train_idx, test_idx in LeaveOneOut().split(X):
        # TODO: fit SelectKBest(f_classif, k=k) on the TRAINING data only
        #       hint: SelectKBest(f_classif, k=k).fit(X[train_idx], y[train_idx])
        selector = None

        Xtr = selector.transform(X[train_idx])
        Xte = selector.transform(X[test_idx])
        scaler = StandardScaler().fit(Xtr)
        model = LogisticRegression(max_iter=1000).fit(scaler.transform(Xtr), y[train_idx])
        proba[test_idx[0]] = model.predict_proba(scaler.transform(Xte))[0, 1]
    return roc_auc_score(y, proba)

auc_nested = loocv_nested(Xbig, ybig)
print(f"Way B (nested — select inside each fold):  AUC = {auc_nested:.3f}")
cu.check(auc_nested < auc_leaky - 0.1,
         f"Leaky {auc_leaky:.2f} vs honest {auc_nested:.2f} — you exposed the leak!",
         "Fit SelectKBest on X[train_idx], y[train_idx] inside the loop.")

## 5. The lesson
The leaky pipeline reported a gorgeous AUC — but it was **fake**. By peeking at the
answers to choose features, the test subject helped pick its own features. The
honest, nested version reveals the truth: with that noisy pile, the score collapses
toward chance.

> **Why our Notebook-13 model is still trustworthy:** its 5 features come from a
> *theory* (central gain → onset response), not from fishing the data. And it
> **survives the permutation test** (section 3). The study reports a fully nested
> version too (AUC ~0.79) as the rigorous number — slightly lower than the fixed-5,
> exactly because the fixed-5 mildly peeked.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
bars = ax.bar(["Leaky\n(peek at answers)", "Honest\n(nested)"],
              [auc_leaky, auc_nested], color=["#CC79A7", cu.CTRL_COLOR])
ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8, label="chance")
for b, a in zip(bars, [auc_leaky, auc_nested]):
    ax.text(b.get_x()+b.get_width()/2, a+0.01, f"{a:.2f}", ha="center", fontweight="bold")
ax.set_ylabel("AUC"); ax.set_ylim(0, 1)
ax.set_title("Same data, same features — leakage makes a fake winner")
ax.legend()
plt.tight_layout()
plt.savefig(cu.save_path("tier3_leakage.png"), dpi=300, bbox_inches="tight")
plt.show()

## 6. The ROC curve
Finally, the picture of our **honest, theory-driven** model. The ROC curve plots
the trade-off between catching EXP cases (true-positive rate) and false alarms
(false-positive rate) as the decision threshold slides. A curve bowing toward the
top-left = a good classifier; the area under it is the AUC.

In [ ]:
real_auc, true_labels, pred_probs = cu.loocv_auc(X, y, return_predictions=True)
fpr, tpr, thresholds = roc_curve(true_labels, pred_probs)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, color=cu.EXP_COLOR, linewidth=2.5, label=f"AUC = {real_auc:.2f}")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="chance")
ax.set_xlabel("False positive rate (false alarms)")
ax.set_ylabel("True positive rate (EXP caught)")
ax.set_title("ROC curve — honest ear-EEG classifier")
ax.legend(loc="lower right")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(cu.save_path("tier3_roc.png"), dpi=300, bbox_inches="tight")
plt.show()

## 🎯 Wrap-up (Tier 3)
You did what separates real science from hype. You didn't just report a number —
you **proved it wasn't luck** (permutation) and **proved it wasn't leakage**
(nested selection). Those two checks are why anyone should believe an ML result on
a small clinical sample.

**For your poster:** state the AUC, the permutation p-value, **and** the sample
size (n=28) — and mention that you guarded against leakage. Honesty about limits is
a strength, not a weakness.